In [2]:
from tensorflow import keras
import pandas as pd
import itertools
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [3]:
def prepare_train_data(input_data, output_data):

    input_seqs = []
    output_seqs = []
    input_chars = set()
    output_chars = set()

    # iterate over all the books
    for seq in range(len(input_data)): 
      
        #if len(output_data[seq]) > 40:
        #  continue
          
        if "*" in input_data[seq]: # cases of ketiv/qere are complicated, just skip them!
          continue

        input_list = list(input_data[seq])

        output_list = list(output_data[seq])
        #output_list = ['\t'] + output_list + ['\n']

        input_seqs.append(input_list)
        output_seqs.append(output_list)

        for input_ch in input_list:
            input_chars.add(input_ch)
        
        for output_ch in output_list:
            output_chars.add(output_ch)
                
    
    input_chars = sorted(list(input_chars))
    output_chars = sorted(list(output_chars))
    
    max_len_input = max([len(seq) for seq in input_seqs])
    max_len_output = max([len(seq) for seq in output_seqs])
    
    # shuffle the data. The model will get the data in small batches, it is preferable if the batches are more or less homogeneous
    # of course the inputs and outputs have to be shuffled identically
    input_seqs, output_seqs = shuffle(input_seqs, output_seqs)
    
    return input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output

In [4]:
colnames = ['book', 'chapter', 'verse', 'text']

input_file = 'ssi_morphology/t-in_voc'
input_df = pd.read_csv(input_file, sep='\t', header=None)
input_df.columns = colnames
input_df.head()

,book,chapter,verse,text
0,Gen,1,1,B.:R;>CIJT B.@R@> >:ELOHIJM >;T HAC.@MAJIM W:>...
1,Gen,1,2,W:H@>@REY H@J:T@H TOHW. W@BOHW. W:XOCEK: <AL P...
2,Gen,1,3,WAJ.O>MER >:ELOHIJM J:HIJ >OWR WAJ:HIJ >OWR
3,Gen,1,4,WAJ.AR:> >:ELOHIJM >ET H@>OWR K.IJ VOWB WAJ.AB...
4,Gen,1,5,WAJ.IQ:R@> >:ELOHIJM L@>OWR JOWM W:LAXOCEK: Q@...


In [5]:
output_file = 'ssi_morphology/t-out'
output_df = pd.read_csv(output_file, sep='\t', header=None)
output_df.columns = colnames
output_df.head()

,book,chapter,verse,text
0,Gen,1,1,B-R>CJT/ BR>[ >LH(J(M/JM >T H-CMJ(M/(JM W->T H...
1,Gen,1,2,W-H->RY/:a HJ(H[&TH THW/ W-BHW/ W-XCK/ <L PN(H...
2,Gen,1,3,W:n-!J!>MR[ >LH(J(M/JM !J!HJ(H[ >WR/ W:n-!J!HJ...
3,Gen,1,4,W:n-!J!R>(H[ >LH(J(M/JM >T H->WR/ KJ VWB[ W:n-...
4,Gen,1,5,W:n-!J!QR>[ >LH(J(M/JM L-(H->WR/ JWM/ W-L-(H-X...


In [6]:
input_df['words'] = input_df.text.str.split('[ \_]')
output_df['words'] = output_df.text.str.split('[ \_]')

In [7]:
words_input = list(itertools.chain(*input_df.words.values))
words_output = list(itertools.chain(*output_df.words.values))

print(len(words_input), len(words_output))

300676 300676


In [8]:
# Some more preprocessing
words_output = [s.replace("=", "").replace(":a", "a").replace(":c", "c").replace(":d", "d").replace(":du", "du") for s in words_output]

In [9]:
print('Max input length:', max([len(s) for s in words_input]))
print('Max output length:', max([len(s) for s in words_output]))

Max input length: 23
Max output length: 30


In [10]:
max_seq_length = 30
words_input_padded = [s.ljust(max_seq_length, ' ') for s in words_input]
words_output_padded = [s.ljust(max_seq_length, ' ') for s in words_output]

In [11]:
input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output = prepare_train_data(words_input_padded, words_output_padded)
print(len(input_seqs))

299488


In [12]:
input_seqs[0]

['>',
 ':',
 'A',
 'C',
 'E',
 'R',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ']

In [13]:
input_char2idx = {c: i for i,c in enumerate(input_chars)}
output_char2idx = {c: i for i,c in enumerate(output_chars)}

In [14]:
input_seqs_num = [[input_char2idx[c] for c in s] for s in input_seqs]
output_seqs_num = [[output_char2idx[c] for c in s] for s in output_seqs]

In [15]:
input_seqs_onehot = keras.utils.to_categorical(input_seqs_num)
input_seqs_onehot.shape

(299488, 30, 33)

In [16]:
output_seqs_onehot = keras.utils.to_categorical(output_seqs_num)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(input_seqs_onehot,output_seqs_onehot, test_size=0.1, random_state = 42)

In [18]:
X_train.shape, y_train.shape

((269539, 30, 33), (269539, 30, 41))

## Create model

In [27]:
import tensorflow_addons as tfa
import numpy as np
import tensorflow as tf

In [104]:
potentials = tf.constant(np.random.random((128, 30, 41)))
y_true = y_train[:128]
potentials.shape, y_true.shape

(TensorShape([128, 30, 41]), (128, 30, 41))

In [105]:
y_true_tags = tf.argmax(y_true, axis=-1)
y_true_tags.shape

TensorShape([128, 30])

In [107]:
log_likelihood, transition_params = tfa.text.crf_log_likelihood(inputs=potentials, # [batch_size, max_seq_length, num_tags]
                                       tag_indices = tf.argmax(y_true, axis=-1), # [batch_size, max_seq_length]
                                       sequence_lengths = tf.repeat(y_true.shape[1], 128) # [batch_size]
                                      )

In [113]:
tf.math.reduce_mean(log_likelihood)

<tf.Tensor: shape=(), dtype=float64, numpy=-113.70264308300804>

In [117]:
text_lens = tf.math.reduce_sum(tf.math.reduce_sum(tf.cast(tf.math.not_equal(X_train[:10], 0), dtype=tf.int32), axis=-1), axis=-1)
text_lens

<tf.Tensor: shape=(10,), dtype=int32, numpy=array([30, 30, 30, 30, 30, 30, 30, 30, 30, 30], dtype=int32)>

## Try 1: subset Model class

In [244]:
class LSTMCRF(tf.keras.Model):
    def __init__(self, bilstm_size, lstm_size, label_size):
        super(LSTMCRF, self).__init__()
        self.bilstm_size = bilstm_size
        self.lstm_size = lstm_size
        self.label_size = label_size

        self.bilstm = keras.layers.Bidirectional(tf.keras.layers.LSTM(bilstm_size, return_sequences=True, dropout=0.5))
        self.lstm = keras.layers.LSTM(units=lstm_size, return_sequences=True)
        self.dense = keras.layers.TimeDistributed(keras.layers.Dense(label_size, activation="relu"))

        self.transition_params = tf.Variable(tf.random.uniform(shape=(label_size, label_size)))

    # @tf.function
    def call(self, inputs, labels=None, training=None):
        sequence_lengths = tf.math.reduce_sum(tf.math.reduce_sum(tf.cast(tf.math.not_equal(inputs, 0), dtype=tf.int32), axis=-1), axis=-1)
        # -1 change 0
        x = self.bilstm(inputs, training=training)
        x = self.lstm(inputs, training=training)
        x = self.dense(x)
        logits = tfa.text.crf.crf_decode(
            x, self.transition_params, sequence_lengths
        )

        if labels is not None:
            label_tags = tf.argmax(labels, axis=-1)
            log_likelihood, self.transition_params = tfa.text.crf_log_likelihood(logits,
                                                                                   label_tags,
                                                                                   sequence_lengths,
                                                                                   transition_params=self.transition_params)
            loss = tf.math.reduce_mean(-log_likelihood)
            self.add_loss(loss)
            return logits, text_lens, loss
        else:
            return logits, text_lens
        
    def _compute_loss(self, labels, prediction, sequence_lengths):
        label_tags = tf.argmax(labels, axis=-1)
        log_likelihood, self.transition_params = tfa.text.crf_log_likelihood(predictions,
                                                                               label_tags,
                                                                               sequence_lengths,
                                                                               transition_params=self.transition_params)
        loss = tf.math.reduce_mean(-log_likelihood)
        
    def train_step(self, data):
        # Mostly copied from https://github.com/luozhouyang/keras-crf/blob/main/keras_crf/crf_model.py
        x, y = data
        with tf.GradientTape() as tape:
            # prediction is potentials
            prediction, text_lens, loss = self(x, y, training=True)
            crf_loss = self._compute_loss(y, prediction, text_lens)
            total_loss = crf_loss + sum(self.losses)
        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Update metrics (includes the metric that tracks the loss)
        y_pred = tf.argmax(prediction, axis=-1)
        self.compiled_metrics.update_state(y, y_pred)
        # Return a dict mapping metric names to current value
        results = {'loss': total_loss, 'crf_loss': crf_loss}
        results.update({m.name: m.result() for m in self.metrics})
        return results

In [245]:
model = LSTMCRF(50, 50, len(output_chars))
model.build(X_train.shape)

In [246]:
model.summary()

Model: "lstmcrf_13"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
bidirectional_33 (Bidirectio multiple                  33600     
_________________________________________________________________
lstm_67 (LSTM)               multiple                  16800     
_________________________________________________________________
time_distributed_33 (TimeDis multiple                  2091      
Total params: 54,172
Trainable params: 54,172
Non-trainable params: 0
_________________________________________________________________


In [247]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    metrics=['acc'])

In [248]:
history = model.fit(X_train[:1000], y_train[:1000], batch_size=256, epochs=2, validation_split=0.1, verbose=1)

Epoch 1/2


/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py:376: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  return py_builtins.overload_of(f)(*args)


TypeError: in user code:

    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/keras/engine/training.py:858 train_function  *
        return step_function(self, iterator)
    <ipython-input-142-a89d8a8085f4>:27 call  *
        log_likelihood, self.transition_params = tfa.text.crf_log_likelihood(logits,
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow_addons/text/crf.py:227 crf_log_likelihood  *
        inputs = tf.convert_to_tensor(inputs)
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/util/dispatch.py:206 wrapper  **
        return target(*args, **kwargs)
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/framework/ops.py:1430 convert_to_tensor_v2_with_dispatch
        return convert_to_tensor_v2(
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/framework/ops.py:1436 convert_to_tensor_v2
        return convert_to_tensor(
    /home/dafne/anac

## Try 2: subsetting layer class

In [150]:
y_train_tags = tf.argmax(y_train, axis=-1)
y_train_tags.shape

TensorShape([269539, 30])

In [184]:
ones = tf.cast(tf.math.not_equal(potentials, 0), dtype=tf.int32)
return tf.reduce_max(tf.reduce_sum(ones, axis=-1), axis=-1)

<tf.Tensor: shape=(128,), dtype=int32, numpy=
array([41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 41, 41, 41, 41], dtype=int32)>

In [181]:
potentials.shape

TensorShape([128, 30, 41])

In [168]:
input_energy_shape = tf.shape(X_train[:10])
raw_input_shape = tf.slice(input_energy_shape, [0], [2])
alt_mask = tf.ones(raw_input_shape)

sequence_length = tf.reduce_sum(tf.cast(alt_mask, tf.int64), 1)
sequence_length

<tf.Tensor: shape=(10,), dtype=int64, numpy=array([30, 30, 30, 30, 30, 30, 30, 30, 30, 30])>

In [193]:
class CRF(tf.keras.layers.Layer):
    def __init__(self, label_size):
        super(CRF, self).__init__()
        self.label_size = label_size
        
    def build(self, input_shape: tf.TensorShape) -> None:
        self.transition_params = self.add_weight(
            shape=(self.label_size, self.label_size),
            regularizer=tf.keras.regularizers.l2(0.1),
            name="transitions",
        )
        self.built = True
        
    def get_sequence_length(self, inputs):
        ones = tf.cast(tf.math.not_equal(inputs, 0), dtype=tf.int32)
        return tf.reduce_max(tf.reduce_sum(ones, axis=-1), axis=-1)
    
    def call(self, potentials, labels=None):
        # Potentials: shape [batch_size, max_seq_len, num_tags]
        
        sequence_lengths = self.get_sequence_length(potentials)
        
        # decode_tags: [batch_size, max_seq_len] 
        # best_score: [batch_size]
        decode_tags, best_score = tfa.text.crf.crf_decode(
            potentials, self.transition_params, sequence_lengths
        )
        
        decode_tags = tf.cast(decode_tags, tf.float64)
        
        if labels is not None:
            label_tags = tf.argmax(labels, axis=-1)
            label_tags = tf.argmax(labels, axis=-1)
            log_likelihood, self.transition_params = tfa.text.crf_log_likelihood(logits,
                                                                                   label_tags,
                                                                                   sequence_lengths,
                                                                                   transition_params=self.transition_params)
            loss = tf.math.reduce_mean(-log_likelihood)
            self.add_loss(loss)
        return decode_tags

In [194]:
inp = keras.layers.Input(shape=(X_train.shape[1:]))
x = keras.layers.Bidirectional(keras.layers.LSTM(units=50, 
                                                 dropout=0.5, 
                                                 return_sequences=True))(inp)
x = keras.layers.LSTM(units=50,
         return_sequences=True)(x)
x = keras.layers.TimeDistributed(keras.layers.Dense(len(output_chars), activation="relu"))(x) 
crf = CRF(len(output_chars))
decoded_sequence = crf(x)
model = keras.Model(inp, decoded_sequence)

/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py:376: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  return py_builtins.overload_of(f)(*args)


In [195]:
model.summary()

Model: "model_10"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_15 (InputLayer)        [(None, 30, 33)]          0         
_________________________________________________________________
bidirectional_23 (Bidirectio (None, 30, 100)           33600     
_________________________________________________________________
lstm_47 (LSTM)               (None, 30, 50)            30200     
_________________________________________________________________
time_distributed_23 (TimeDis (None, 30, 41)            2091      
_________________________________________________________________
crf_13 (CRF)                 (None, 30)                1681      
Total params: 67,572
Trainable params: 67,572
Non-trainable params: 0
_________________________________________________________________


In [196]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    metrics=['acc'],
loss='categorical_crossentropy')

In [197]:
model.losses

[<tf.Tensor: shape=(), dtype=float32, numpy=4.3110375>]

In [198]:
model.predict(X_train[:10])

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


array([[38., 38., 38., 38., 38., 38., 38., 38., 38., 38., 38., 38., 38.,
        38., 38., 38., 38., 15., 38., 15., 25., 21., 34., 25., 21., 34.,
        34., 21., 25., 34.],
       [38., 38., 38., 15., 20., 20., 20., 20., 21., 25., 21., 34., 25.,
        21., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25.,
        34., 34., 25., 34.],
       [38., 38., 38., 38., 38., 38., 38., 38., 15., 38., 15., 25., 21.,
        34., 25., 21., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25.,
        34., 34., 25., 34.],
       [38., 38., 38., 38., 38., 38., 38., 38., 38., 38., 15., 25., 21.,
        34., 25., 21., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25.,
        34., 34., 25., 34.],
       [38., 38., 38., 38., 38., 38., 38., 38., 38., 15., 25., 21., 34.,
        25., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25., 34., 25.,
        34., 34., 25., 34.],
       [38., 38., 38., 38., 38., 38., 15., 20., 20., 20., 20., 20., 33.,
        10., 21., 34., 25., 21., 34., 25., 34., 25.,

In [199]:
y_train_tags[:10]

<tf.Tensor: shape=(10, 30), dtype=int64, numpy=
array([[27,  5, 18,  5,  9, 18, 16, 18,  6, 16,  4, 15,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [18,  5, 10, 20,  6, 16,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [27,  7, 36,  5,  1, 16,  1, 10,  3, 27,  9, 31,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [15,  5, 28, 18, 10,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 9, 11, 23,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [16, 13, 23,  9, 18,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [27,  7, 36,  5,  1, 16,  1,  3, 20, 25, 20, 31,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0

In [200]:
history = model.fit(X_train[:1000], y_train_tags[:1000], batch_size=256, epochs=2, validation_split=0.1, verbose=1)

Epoch 1/2
4/4 [==============================] - ETA: 0s - loss: 357.8534 - acc: 0.0833WARNING:tensorflow:AutoGraph could not transform <function Model.make_test_function.<locals>.test_function at 0x7fcb8db08d30> and will run it as-is.
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
4/4 [==============================] - 6s 1s/step - loss: 357.8534 - acc: 0.08

InvalidArgumentError:  seq_lens(89) > input.dims(1)
	 [[node model_10/crf_13/ReverseSequence (defined at /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow_addons/text/crf.py:571) ]] [Op:__inference_train_function_89843]

Errors may have originated from an input operation.
Input Source operations connected to node model_10/crf_13/ReverseSequence:
 model_10/crf_13/Maximum (defined at /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow_addons/text/crf.py:563)	
 model_10/crf_13/rnn/transpose_2 (defined at /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow_addons/text/crf.py:496)

Function call stack:
train_function


## Try 3: using tfa.layers.CRF layer

In [204]:
inp = keras.layers.Input(shape=(X_train.shape[1:]))
x = keras.layers.Bidirectional(keras.layers.LSTM(units=50, 
                                                 dropout=0.5, 
                                                 return_sequences=True))(inp)
x = keras.layers.LSTM(units=50,
         return_sequences=True)(x)
x = keras.layers.TimeDistributed(keras.layers.Dense(len(output_chars), activation="relu"))(x) 
crf = tfa.layers.CRF(len(output_chars))
decoded_sequence, potentials, sequence_length, chain_kernel = crf(x)
model = keras.Model(inp, potentials)

/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py:376: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  return py_builtins.overload_of(f)(*args)


In [205]:
model.summary()

Model: "model_12"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_17 (InputLayer)        [(None, 30, 33)]          0         
_________________________________________________________________
bidirectional_25 (Bidirectio (None, 30, 100)           33600     
_________________________________________________________________
lstm_51 (LSTM)               (None, 30, 50)            30200     
_________________________________________________________________
time_distributed_25 (TimeDis (None, 30, 41)            2091      
_________________________________________________________________
crf_15 (CRF)                 [(None, 30), (None, 30, 4 3485      
Total params: 69,376
Trainable params: 69,376
Non-trainable params: 0
_________________________________________________________________


In [206]:
#Optimiser 
adam = keras.optimizers.Adam(learning_rate=0.0005, beta_1=0.9, beta_2=0.999)
batch_size = 128
def crf_loss(potentials, y_true):
    log_likelihood, transition_params = tfa.text.crf_log_likelihood(inputs=potentials, # [batch_size, max_seq_length, num_tags]
                                       tag_indices = tf.argmax(y_true, axis=-1), # [batch_size, max_seq_length]
                                       sequence_lengths = tf.repeat(y_true.shape[1], batch_size) # [batch_size]
                                      )
    loss = tf.math.reduce_mean(-log_likelihood)
    loss += keras.losses.categorical_crossentropy(y_true, potentials)
    return loss
# Compile model
model.compile(optimizer=adam, metrics=[tfa.text.crf_binary_score], loss=crf_loss)

In [209]:
crf_loss(tf.constant(np.random.random(y_true.shape)), y_true)

<tf.Tensor: shape=(128, 30), dtype=float64, numpy=
array([[121.04719468, 122.38080198, 121.11313951, ..., 121.65207002,
        121.11057531, 121.47427278],
       [121.01966589, 121.7258518 , 122.472163  , ..., 122.97271014,
        121.09555101, 121.19648912],
       [122.99605752, 125.76689232, 120.94769567, ..., 122.1495082 ,
        121.22334069, 121.85655202],
       ...,
       [123.32206884, 121.46030917, 121.399529  , ..., 121.30103723,
        121.52129233, 121.35186804],
       [122.00783144, 121.76166491, 122.75710735, ..., 121.73559469,
        120.98558392, 122.3756816 ],
       [122.79428995, 122.36807531, 121.82238424, ..., 121.22651801,
        122.13541941, 122.11027404]])>

In [210]:
history = model.fit(X_train[:1000], y_train[:1000], batch_size=batch_size, epochs=2, validation_split=0.1, verbose=1)

Epoch 1/2


/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py:376: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  return py_builtins.overload_of(f)(*args)


TypeError: in user code:

    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/keras/engine/training.py:858 train_function  *
        return step_function(self, iterator)

    TypeError: tf__crf_binary_score() missing 1 required positional argument: 'transition_params'


### Try 4: use keras_crf package

In [211]:
from keras_crf import CRFModel

In [221]:
inp = keras.layers.Input(shape=(X_train.shape[1:]))
x = keras.layers.Bidirectional(keras.layers.LSTM(units=50, 
                                                 dropout=0.5, 
                                                 return_sequences=True))(inp)
x = keras.layers.LSTM(units=50,
         return_sequences=True)(x)
x = keras.layers.TimeDistributed(keras.layers.Dense(len(output_chars), activation="relu"))(x) 
#outputs = tf.argmax(x, -1)
outputs = x
base = keras.Model(inputs=inp, outputs=outputs)
model = CRFModel(base, len(output_chars))
base.summary()

Model: "model_16"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_21 (InputLayer)        [(None, 30, 33)]          0         
_________________________________________________________________
bidirectional_29 (Bidirectio (None, 30, 100)           33600     
_________________________________________________________________
lstm_59 (LSTM)               (None, 30, 50)            30200     
_________________________________________________________________
time_distributed_29 (TimeDis (None, 30, 41)            2091      
Total params: 65,891
Trainable params: 65,891
Non-trainable params: 0
_________________________________________________________________


In [223]:
model.build(tf.TensorShape([None, 30, 33]))
model.summary()

Model: "crf_model_7"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
model_16 (Functional)        (None, 30, 41)            65891     
_________________________________________________________________
crf_19 (CRF)                 multiple                  3485      
Total params: 69,376
Trainable params: 69,376
Non-trainable params: 0
_________________________________________________________________


/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py:376: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  return py_builtins.overload_of(f)(*args)


In [224]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    metrics=['acc'])

In [228]:
prediction = model.call(X_train[:2], training=True)
crf_loss = model._compute_loss(prediction, y_train_tags[:2])

/home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow_addons/text/crf.py:545: UserWarning: CRF decoding models have serialization issues in TF >=2.5 . Please see isse #2476
  warnings.warn(


IndexError: list index out of range

In [225]:
history = model.fit(X_train[:1000], y_train[:1000], batch_size=256, epochs=2, validation_split=0.1, verbose=1)

Epoch 1/2


ValueError: in user code:

    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/keras/engine/training.py:858 train_function  *
        return step_function(self, iterator)
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/keras/engine/training.py:847 step_function  **
        self.train_function = lambda iterator: self._cluster_coordinator.schedule(  # pylint: disable=g-long-lambda
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/distribute/distribute_lib.py:1286 run
        return self._extended.call_for_each_replica(fn, args=args, kwargs=kwargs)
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/distribute/distribute_lib.py:2849 call_for_each_replica
        return self._call_for_each_replica(fn, args, kwargs)
    /home/dafne/anaconda3/envs/hebrew/lib/python3.9/site-packages/tensorflow/python/distribute/distribute_lib.py:3632 _call_for_each_replica
        return fn(*args, **kwargs)
    /